# 🚀 AI Multi-Docs Extraction Pipeline: Step-by-Step Walkthrough (Per-Batch)

สมุดบันทึก (Jupyter Notebook) สำหรับทดสอบกระบวนการสกัดและประมวลผลเอกสาร **ทีละขั้นตอน (Step-by-Step Execution & Observability)** ในระดับ **`batch_id`**
- 📂 **ชุดข้อมูลทดสอบ**: สกัด 3 หน้าแรกจาก `Grab_202606_000007.pdf` เป็นไฟล์ `Grab_Sample_3Pages.pdf` จัดเก็บไว้ใต้ `01_drop_zone/Test_Walkthrough/`
- 🛡️ **ระบบ Per-Batch Isolation**: ทุกขั้นตอน (Stage 3 ➔ 4 ➔ 5) กำหนดให้ใช้ `batch_id` เป็นแกนหลัก เพื่อป้องกัน Process ชนกัน 100%

## 🛠️ Step 0: ตั้งค่า Working Directory, นำเข้าโมดูล และเตรียมไฟล์ทดสอบ

In [2]:
import os
import sys
import glob
import json
import shutil
import pandas as pd
from IPython.display import display, JSON
from dotenv import load_dotenv

# 1. Ensure working directory is set to project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"📂 Project Root Working Directory: {os.getcwd()}")

# 2. Load Environment Variables & Pipeline Services
load_dotenv()
from src.application.pipeline import (
    init_system,
    split_and_match,
    extract_documents,
    async_extract_documents,
    validate_documents,
    transform_to_db,
    reset_pipeline_data,
    release_pending_merchant_files,
)
from src.infrastructure.core.config import load_system_settings, get_default_doc_type, get_default_company_code
from src.infrastructure.external.storage.storage_manager import storage_manager
from src.infrastructure.database import (
    get_pending_merchants,
    approve_merchant,
    get_db_session,
    ExpenseReceipt,
    ExpenseReceiptItem,
)
from sqlalchemy import select, func

DOC_TYPE = get_default_doc_type()
COMPANY_CODE = get_default_company_code()
ACTIVE_BATCH_ID = None
print(f"✅ Pipeline Services Ready. Active Doc Type: '{DOC_TYPE}' | Company: '{COMPANY_CODE}'")

from src.infrastructure.core import set_current_user_id, SystemUserId
set_current_user_id(SystemUserId.SYSTEM_TEST)
print(f"🔑 UserContext Active: '{SystemUserId.SYSTEM_TEST}' (System Test Runner)")

# 3. Ensure Test Fixture in Test_Walkthrough drop zone
walkthrough_drop_dir = storage_manager.get_drop_zone_dir(COMPANY_CODE, DOC_TYPE, "Test_Walkthrough")
os.makedirs(walkthrough_drop_dir, exist_ok=True)
test_pdf_target = os.path.join(walkthrough_drop_dir, "Grab_Sample_3Pages.pdf").replace("\\", "/")
master_fixture = "tests/fixtures/sample_docs/Grab_Sample_3Pages.pdf"

if not os.path.exists(test_pdf_target) and os.path.exists(master_fixture):
    shutil.copy(master_fixture, test_pdf_target)
    print(f"📋 Copied sample test file to Drop Zone: {test_pdf_target}")
else:
    print(f"📋 Test File Ready at: {test_pdf_target}")

📂 Project Root Working Directory: d:\PROKUNG\GitHub-Source\AI-Multi-Docs-Extraction-Pipeline
✅ Pipeline Services Ready. Active Doc Type: 'expense_receipt' | Company: 'C00000_SAMPLE'
🔑 UserContext Active: 'usr_system_test' (System Test Runner)
📋 Test File Ready at: storage/companies/C00000_SAMPLE/expense_receipt/01_drop_zone/Test_Walkthrough/Grab_Sample_3Pages.pdf


## 🧹 Step 0.1: (Optional) รีเซ็ตระบบจาก 0 (Drop Database & Clean Fresh Start)
กดรันเซลล์นี้เมื่อต้องการ **Drop Database SQLite ทิ้งแล้ว Re-seed Master Data ใหม่จาก 0** พร้อมทั้ง **ล้างไฟล์ชั่วคราวใน `03_preprocess/` และ `04_processing/`** และเตรียมไฟล์ตัวอย่างใหม่ใน `Test_Walkthrough/` เพื่อเริ่มทดสอบใหม่ตั้งแต่ต้น

In [3]:
# รีเซ็ต Pipeline Storage Temp และ Database สำหรับรอบการทดสอบใหม่ (เริ่มจาก 0)
reset_result = reset_pipeline_data(doc_type=DOC_TYPE, clear_storage_temp=True, clear_database=True)
print("🧹 Reset Pipeline Data Result:", reset_result)

# นำเข้าไฟล์ตัวอย่าง Grab 3 หน้าใหม่หากยังไม่มีใน Drop Zone
if os.path.exists(master_fixture) and not os.path.exists(test_pdf_target):
    shutil.copy(master_fixture, test_pdf_target)
    print(f"🔄 Restored Test Fixture: {test_pdf_target}")

print("🎉 Ready for a brand new clean step-by-step walkthrough run!")

[2026-08-29 21:26:50] [INFO] Resetting Pipeline Data (Fresh Start)
[2026-08-29 21:26:50] [INFO] Dropping and recreating all database tables...
[2026-08-29 21:26:51] [INFO] Relational database schema initialized successfully via SQLAlchemy Base metadata.
[2026-08-29 21:26:51] [INFO] Database master data seeding completed successfully across all 8 tables.
[2026-08-29 21:26:51] [INFO] Cleaned 0 temporary files across raw_data, preprocess, and processing queues.


🧹 Reset Pipeline Data Result: {'database_reset': True, 'storage_cleaned': True, 'deleted_files_count': 0}
🎉 Ready for a brand new clean step-by-step walkthrough run!


## ⚙️ Step 1: System Initialization (`Run_01`)
ตรวจสอบความพร้อมของไฟล์คอนฟิก `settings.json`, Schema ฐานข้อมูล SQLite, และโครงสร้างโฟลเดอร์ใน `storage/`

In [4]:
print("--- [Stage 1] Initializing System & Validating Environment ---")
init_success = init_system(drop_and_recreate=False)
if init_success:
    print("🎉 System is READY and all storage folders & DB tables are verified!")
else:
    print("❌ System initialization encountered errors. Please check configs or .env")

[2026-08-29 21:26:53] [INFO] Starting Stage 1 (Init): System Initialization & Health Check
[2026-08-29 21:26:53] [INFO] [1/4] Checking Central settings.json...
[2026-08-29 21:26:53] [INFO] [PASS] settings.json is valid and complete.
[2026-08-29 21:26:53] [INFO] [2/4] Checking DocType-specific configurations...
[2026-08-29 21:26:53] [INFO]   * Checking doc_type 'expense_receipt'...
[2026-08-29 21:26:53] [INFO]     [PASS] DocType 'expense_receipt' configs are valid.
[2026-08-29 21:26:53] [INFO]   * Checking doc_type 'tax_invoice'...
[2026-08-29 21:26:53] [INFO]     [PASS] DocType 'tax_invoice' configs are valid.
[2026-08-29 21:26:53] [INFO]   * Checking doc_type 'withholding_tax'...
[2026-08-29 21:26:53] [INFO]     [PASS] DocType 'withholding_tax' configs are valid.
[2026-08-29 21:26:53] [INFO] [3/4] Checking Environment & Package Dependencies...


--- [Stage 1] Initializing System & Validating Environment ---


[2026-08-29 21:26:54] [INFO] [PASS] All required Python packages are installed.
[2026-08-29 21:26:54] [INFO] [4/4] Initializing Pipeline Storage Directories & DB Schema...
[2026-08-29 21:26:54] [INFO] Relational database schema initialized successfully via SQLAlchemy Base metadata.
[2026-08-29 21:26:54] [INFO] Database master data seeding completed successfully across all 8 tables.
[2026-08-29 21:26:54] [INFO] [PASS] Ensured 9 directories are created with .gitkeep.
[2026-08-29 21:26:54] [INFO] [SYSTEM STATUS] System is READY and fully configured!


🎉 System is READY and all storage folders & DB tables are verified!


## 📄 Step 2: Ingest, Classify & Split PDFs (`Run_02`)
อ่านไฟล์เอกสารตัวอย่าง `Grab_Sample_3Pages.pdf` จาก `01_drop_zone/Test_Walkthrough/` เพื่อ:
1. ตรวจจับร้านค้า (Merchant Matching: Fast Prefix / Ingestion Classification)
2. ตัดหน้า PDF เป็นไฟล์ภาพ `.jpg` 3 หน้าลงใน `03_preprocess/`
3. ลงทะเบียน Batch & Pages เข้าสู่ฐานข้อมูล SQLite และส่งออก `batch_id`

In [5]:
print("--- [Stage 2] Ingesting Documents from Drop Zone ---")
split_results = split_and_match(doc_type=DOC_TYPE, input_file=test_pdf_target, company_code=COMPANY_CODE)

if split_results:
    ACTIVE_BATCH_ID = split_results[0].get("batch_id")
    print(f"\n✅ Successfully processed batch:")
    print(f"📦 ACTIVE BATCH ID: {ACTIVE_BATCH_ID}")
    print(f"   - Matched Merchant: {split_results[0].get('matched_source')}")
    print(f"   - Total Pages: {split_results[0].get('total_pages')}")
    print(f"   - Split Page Images ({len(split_results[0].get('page_images', []))} files):")
    for img in split_results[0].get('page_images', []):
        print(f"     🖼️ {img}")
else:
    print("ℹ️ File held in PENDING awaiting merchant approval. (See Step 2.1 below)")

[2026-08-29 21:26:56] [INFO] Starting Stage 2 (Split & Match): Processing Files & Matching Merchants
[2026-08-29 21:26:56] [INFO] Found 1 document file(s) to process for company 'C00000_SAMPLE'.
[2026-08-29 21:26:56] [INFO] 
--- Processing: Grab_Sample_3Pages.pdf (PDF Document) [Batch: batch_6980c2b6a784] [Company: C00000_SAMPLE] ---


--- [Stage 2] Ingesting Documents from Drop Zone ---


[2026-08-29 21:26:57] [INFO] 🤖 AI Request -> Provider: gemini | Model: gemini-3.5-flash-lite | Images: 1 (Attempt 1/3)
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
[2026-08-29 21:26:59] [INFO] ✅ AI Extraction Completed in 2.17s | Tokens: in=1304, out=65 | Cost: $0.00000 (0.000 THB)
[2026-08-29 21:26:59] [INFO] AI Fast Classifier result: {'tax_id': '0105556090377', 'merchant_name': 'Grabtaxi (Thailand) Co., Ltd.', 'suggested_short_name': 'grab', 'has_tax_id': True}
[2026-08-29 21:26:59] [INFO] Auto-discovered new merchant 'Grabtaxi (Thailand) Co., Ltd.' (ID: merch_00b1691efd53) by 'usr_system_test' -> Gatekeeper status: PENDING
[2026-08-29 21:26:59] [WARNING] Merchant 'Grabtaxi (Thailand) Co., Ltd.' is in PENDING status. File held for review 

ℹ️ File held in PENDING awaiting merchant approval. (See Step 2.1 below)


## 🔍 Step 2.1: Check & Confirm Pending Merchants (Human-in-the-Loop)
หากเอกสารมาจากร้านค้าใหม่ที่ยังไม่อยู่ในระบบ หรือยังไม่ได้รับการอนุมัติ เอกสารจะถูกพักไว้ที่ `02_raw_data/PENDING/`
เซลล์นี้ช่วยให้สามารถตรวจสอบและอนุมัติร้านค้า (`approve_merchant`) พร้อมทั้งปล่อยเอกสาร (`release_pending_merchant_files`) เข้าสู่กระบวนการสกัดข้อมูลต่อไปได้ทันที

In [6]:
print("--- [Stage 2.1] Checking Pending Merchants Waiting for Approval ---")
pending_list = get_pending_merchants()

if pending_list:
    print(f"🔍 Found {len(pending_list)} pending merchant(s):\n")
    for p in pending_list:
        m_id = p.get("merchant_id")
        m_name = p.get("merchant_name")
        tax_id = p.get("tax_id")
        short_name = p.get("short_name") or "grab"
        print(f"📌 Pending Merchant: {m_name} (ID: {m_id}, Tax ID: {tax_id})")
        
        approved, msg = approve_merchant(merchant_id=m_id, approved_by=SystemUserId.SYSTEM_TEST, short_name=short_name)
        print(f"   ➔ Approval Status: {approved} ({msg if not approved else 'Success'})")
        
        released_files = release_pending_merchant_files(doc_type=DOC_TYPE, tax_id=tax_id, short_name=short_name, company_code=COMPANY_CODE)
        if released_files:
            ACTIVE_BATCH_ID = released_files[0].get("batch_id")
            print(f"   ➔ Released Batch ID: {ACTIVE_BATCH_ID} ({len(released_files)} file batch)")
else:
    print("✅ No pending merchants awaiting review.")

print(f"\n🎯 Target Active Batch ID for Stage 3-5: '{ACTIVE_BATCH_ID}'")

[2026-08-29 21:27:11] [INFO] Merchant 'Grabtaxi (Thailand) Co., Ltd.' (merch_00b1691efd53) APPROVED by 'usr_system_test'.
[2026-08-29 21:27:11] [INFO] Moved 1 pending files for merchant '0105556090377_grab' (Company: C00000_SAMPLE) to approved raw data.
[2026-08-29 21:27:11] [INFO] Removed emptied pending staging folder: 'storage/companies/C00000_SAMPLE/expense_receipt/02_raw_data/PENDING/0105556090377_grab'.
[2026-08-29 21:27:11] [INFO] Starting Stage 2 (Split & Match): Processing Files & Matching Merchants
[2026-08-29 21:27:11] [INFO] Found 1 document file(s) to process for company 'C00000_SAMPLE'.
[2026-08-29 21:27:11] [INFO] 
--- Processing: Grab_Sample_3Pages.pdf (PDF Document) [Batch: batch_589eae84ad1b] [Company: C00000_SAMPLE] ---
[2026-08-29 21:27:11] [INFO] ⚡ Zero-Cost Bypass: 'Grab_Sample_3Pages.pdf' matched merchant 'Grabtaxi (Thailand) Co., Ltd.' via prefix (Action: 'PROCEED').
[2026-08-29 21:27:11] [INFO] Splitting PDF: 'storage/companies/C00000_SAMPLE/expense_receipt/02_

--- [Stage 2.1] Checking Pending Merchants Waiting for Approval ---
🔍 Found 1 pending merchant(s):

📌 Pending Merchant: Grabtaxi (Thailand) Co., Ltd. (ID: merch_00b1691efd53, Tax ID: 0105556090377)
   ➔ Approval Status: True (Success)


[2026-08-29 21:27:11] [INFO] [PASS] PDF split validated: Exactly 3/3 page(s) generated as .jpg.
[2026-08-29 21:27:11] [INFO] Registered Batch 'batch_589eae84ad1b' with 3 page(s) as PREPROCESSED.


   ➔ Released Batch ID: batch_589eae84ad1b (1 file batch)

🎯 Target Active Batch ID for Stage 3-5: 'batch_589eae84ad1b'


## 🤖 Step 3: AI Document Extraction (`Run_03`)
นำรูปภาพหน้าที่ตัดแล้วใน `03_preprocess/` ของ **`ACTIVE_BATCH_ID`** ส่งให้ Multimodal AI สกัดข้อมูลตามโครงสร้าง `extract-schema.json`
และบันทึกไฟล์ JSON ที่สกัดได้ลงใน `04_processing/` พร้อมระบบ Smart Checkpointing ป้องกันการเรียก AI ซ้ำ

In [7]:
print(f"--- [Stage 3] Extracting Document Data for Batch: '{ACTIVE_BATCH_ID}' ---")
if not ACTIVE_BATCH_ID:
    raise ValueError("ACTIVE_BATCH_ID is not set. Please run Step 2 / Step 2.1 first.")

extract_result = extract_documents(batch_id=ACTIVE_BATCH_ID, doc_type=DOC_TYPE, company_code=COMPANY_CODE)
print(f"📊 Extraction Summary: {extract_result}")

# Preview extracted JSON payload files
queue_pattern = f"{storage_manager.get_processing_dir(COMPANY_CODE, DOC_TYPE)}/**/*.json"
queue_files = glob.glob(queue_pattern, recursive=True)
if queue_files:
    print(f"\n💾 Found {len(queue_files)} extracted JSON file(s):\n")
    sample_file = queue_files[0]
    print(f"   Showing preview from: {sample_file}")
    with open(sample_file, "r", encoding="utf-8") as jf:
        sample_json = json.load(jf)
    #display(JSON(sample_json))
else:
    print("ℹ️ No JSON files found in extracted storage.")

[2026-08-29 21:27:31] [INFO] Starting Stage 3 (Extract): Extracting Structured Data via Multimodal AI [Batch: batch_589eae84ad1b]
[2026-08-29 21:27:31] [INFO] Found 1 batch(es) to extract with AI for company 'C00000_SAMPLE'...
[2026-08-29 21:27:31] [INFO] 
--- Extracting Batch: batch_589eae84ad1b (Grab_Sample_3Pages.pdf) | Total Chunks: 1 | Pending Chunks: 1 [Company: C00000_SAMPLE] ---
[2026-08-29 21:27:31] [INFO] Sending AI extraction request for Batch 'batch_589eae84ad1b' Chunk #1 (3 pages)...
[2026-08-29 21:27:31] [INFO] AI Config resolved for doc_type 'expense_receipt' (merchant '0105556090377_grab'): Provider='gemini', Model='gemini-3.5-flash-lite'
[2026-08-29 21:27:31] [INFO] Attempting extraction using credential 'conf_default_provider_free' (env: 'api_key_env_default_free')...


--- [Stage 3] Extracting Document Data for Batch: 'batch_589eae84ad1b' ---


[2026-08-29 21:27:32] [INFO] 🤖 AI Request -> Credential: 'conf_default_provider_free' | Model: gemini-3.5-flash-lite | Images: 3 (Attempt 1/3)
[2026-08-29 21:27:37] [INFO] ✅ AI Extraction OK in 4.88s | Tokens: in=4064, out=1543 | Cost: $0.00000 (0.000 THB)
[2026-08-29 21:27:37] [INFO] Chunk #1 of Batch 'batch_589eae84ad1b' extracted and checkpointed successfully.
[2026-08-29 21:27:37] [INFO] All 1 chunks completed & merged for Batch 'batch_589eae84ad1b'. Status: EXTRACTED.


📊 Extraction Summary: {'success': True, 'batches_processed': 1, 'batches_failed': 0, 'documents_extracted': 1}

💾 Found 3 extracted JSON file(s):

   Showing preview from: storage/companies/C00000_SAMPLE/expense_receipt/04_processing\0105556090377_grab\expense_receipt_no_tax_Grab_Sample_3Pages_batch_58_p1.json


## 🛡️ Step 4: Validate & Post-Process Data (`Run_04`)
ตรวจสอบความถูกต้องของข้อมูลจริงใน `04_processing/` สำหรับ **`ACTIVE_BATCH_ID`**:
- ตรวจสอบเลขประจำตัวผู้เสียภาษี 13 หลัก (Tax ID Validation)
- ปรับรูปแบบวันที่ปี พ.ศ. ➔ ค.ศ. (BE to AD Normalization)
- ตรวจสอบสูตรการเงิน (Subtotal - Discount + VAT == Net)
- ตรวจสอบผลรวมรายการสินค้าเทียบกับ Subtotal
- กำหนดสถานะ `PROCESSED` หรือ `NEEDS_REVIEW`

In [8]:
print(f"--- [Stage 4] Validating Records for Batch: '{ACTIVE_BATCH_ID}' ---")
if not ACTIVE_BATCH_ID:
    raise ValueError("ACTIVE_BATCH_ID is not set. Please run Step 2 / Step 3 first.")

validate_result = validate_documents(batch_id=ACTIVE_BATCH_ID, doc_type=DOC_TYPE, company_code=COMPANY_CODE)
print(f"📊 Validation Summary: {validate_result}")

[2026-08-29 21:27:42] [INFO] Starting Stage 4 (Validation & Post-Processing) UseCase [Batch: batch_589eae84ad1b]
[2026-08-29 21:27:42] [INFO] No open documents found for validation in batch 'batch_589eae84ad1b'.


--- [Stage 4] Validating Records for Batch: 'batch_589eae84ad1b' ---
📊 Validation Summary: {'validated': 0, 'auto_approved': 0, 'needs_review': 0}


## 💾 Step 5: Transform Data to Relational SQLite Database (`Run_05`)
นำเข้าข้อมูลที่ผ่านการตรวจสอบแล้วของ **`ACTIVE_BATCH_ID`** เข้าสู่ฐานข้อมูล SQLite ในตาราง `document_controls`, `expense_receipts`, และ `expense_receipt_items`

In [9]:
print(f"--- [Stage 5] Importing Records into Relational DB for Batch: '{ACTIVE_BATCH_ID}' ---")
if not ACTIVE_BATCH_ID:
    raise ValueError("ACTIVE_BATCH_ID is not set. Please run previous steps first.")

db_result = transform_to_db(batch_id=ACTIVE_BATCH_ID, doc_type=DOC_TYPE, company_code=COMPANY_CODE)
print(f"📊 DB Transformation Summary: {db_result}\n")

# Query database records using Pure SQLAlchemy 2.0
with get_db_session() as session:
    receipt_count = session.scalar(select(func.count()).select_from(ExpenseReceipt))
    item_count = session.scalar(select(func.count()).select_from(ExpenseReceiptItem))
    receipts = session.scalars(select(ExpenseReceipt).limit(10)).all()
    
    print(f"📊 Total Receipts in DB: {receipt_count}")
    print(f"📊 Total Line Items in DB: {item_count}")
    print("\n--- [Database Sample Records] ---")
    for rc in receipts:
        print(f"📄 Receipt ID: {rc.receipt_id} | Merchant: {rc.merchant_name} | Net: {rc.net_amount} THB | Date: {rc.transaction_date}")

[2026-08-29 21:27:46] [INFO] Starting Stage 3 (Transform to DB) UseCase [Batch: batch_589eae84ad1b]
[2026-08-29 21:27:46] [INFO] Found 3 page(s) to import into DB relational tables for company 'C00000_SAMPLE'...
[2026-08-29 21:27:46] [INFO] Imported document record 'doc_0e67f878103f' (Company: C00000_SAMPLE, Status: PROCESSED)
[2026-08-29 21:27:46] [INFO] Imported document record 'doc_f46c29bb8ea6' (Company: C00000_SAMPLE, Status: PROCESSED)
[2026-08-29 21:27:46] [INFO] Imported document record 'doc_3d598e222393' (Company: C00000_SAMPLE, Status: PROCESSED)


--- [Stage 5] Importing Records into Relational DB for Batch: 'batch_589eae84ad1b' ---
📊 DB Transformation Summary: {'imported': 3, 'failed': 0}

📊 Total Receipts in DB: 3
📊 Total Line Items in DB: 3

--- [Database Sample Records] ---
📄 Receipt ID: rcpt_0068112ea19a | Merchant: Grabtaxi (Thailand) Co., Ltd. | Net: 2319.55 THB | Date: 2026-06-01
📄 Receipt ID: rcpt_350df6e1d60d | Merchant: Grabtaxi (Thailand) Co., Ltd. | Net: 2237.86 THB | Date: 2026-06-02
📄 Receipt ID: rcpt_d1e02d59910b | Merchant: Grabtaxi (Thailand) Co., Ltd. | Net: 1307.43 THB | Date: 2026-06-03
